# Transfer learning

In this lab we will make use of pretrained models in order to boost performance on smaller datasets. For this experiment, we will be working with an AlexNet model pretrained on the Imagenet dataset in order to get a good accuracy score on the Caltech 101 dataset.

### Prerequisites

1. In order to perform the experiments, please download in advance the Caltech 101 dataset from https://drive.google.com/file/d/137RyRjvTBkBiIfeYBNZBtViDHQ6_Ewsp/view
2. In the working directory please create a folder named 'dataset' and a subfolder named 'caltech101' within it. Extract the dataset in the subfolder. The overall folder structure should look as follows: dataset/caltech101/101_ObjectCategories.
3. Install the torchvision module using 'conda install torchvision' if you have not done so already.

In [ ]:
from tqdm import tqdm
import numpy as np
import torch
import torchvision
import warnings

warnings.filterwarnings('ignore')
NUM_CLASSES = 101

Firstly, we will load the AlexNet model architecture using torchvision. All available models with their respective parameters can be found at: https://pytorch.org/vision/stable/models.html

In [ ]:
model = torchvision.models.alexnet()

In the first run we will just load the model architecture, without the pretrained weights. We can visualize the model architecture as follows:

In [ ]:
model

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

Next, we will load the Caltech 101 dataset and apply the neccesary transformations on it. Afterwards, we will split the dataset into train, validation and test.

In this block of code, define the dataloaders for train, validation and test and try to iterate through the data. What happens? Try to fix the problem using a lambda transform: https://pytorch.org/vision/stable/transforms.html#generic-transforms

In [ ]:
import sys, os, io, zipfile, tarfile, shutil, hashlib, urllib.request
from pathlib import Path

CALTECHDATA_URL = "https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip?download=1"
DEST_ROOT = Path("./dataset")
RAW_DIR   = DEST_ROOT / "raw"
EXTRACT_DIR = DEST_ROOT / "caltech101"
ZIP_PATH = RAW_DIR / "caltech-101.zip"

RAW_DIR.mkdir(parents=True, exist_ok=True)

def md5(path, chunk=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

# 1) Download zip if needed
if not ZIP_PATH.exists():
    print("Downloading caltech-101.zip from CaltechDATA ...")
    urllib.request.urlretrieve(CALTECHDATA_URL, ZIP_PATH)
    print("Saved:", ZIP_PATH)
else:
    print("Already have:", ZIP_PATH)

# Optional integrity note from the page (not enforced): md5 should be 3138e1922a9193bfa496528edbbc45d0
print("md5:", md5(ZIP_PATH))


Saved: dataset/raw/caltech-101.zip
md5: 3138e1922a9193bfa496528edbbc45d0


In [ ]:
# 2) Extract the two inner tar archives into ./dataset/caltech101/
# The zip contains: caltech-101/101_ObjectCategories.tar.gz and Annotations.tar

def safe_extract_tar(tar_path: Path, dest: Path):
    mode = "r:gz" if tar_path.suffixes[-1] == ".gz" else "r:"
    with tarfile.open(tar_path, mode) as t:
        def is_within_directory(directory, target):
            import os
            abs_directory = os.path.abspath(directory)
            abs_target = os.path.abspath(target)
            return os.path.commonprefix([abs_directory, abs_target]) == abs_directory
        for member in t.getmembers():
            target_path = dest / member.name
            if not is_within_directory(dest, target_path):
                raise Exception("Blocked path traversal in tar file")
        t.extractall(dest)

# Clean previous extraction if it looks incomplete
if EXTRACT_DIR.exists():
    # Heuristic: only wipe if missing both target folders
    if not (EXTRACT_DIR / "101_ObjectCategories").exists() or not (EXTRACT_DIR / "Annotations").exists():
        shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Unzip only if we don't already see the two folders
if not (EXTRACT_DIR / "101_ObjectCategories").exists() or not (EXTRACT_DIR / "Annotations").exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        names = z.namelist()
        # Find inner tar paths
        inner_img = [n for n in names if n.endswith("101_ObjectCategories.tar.gz")][0]
        inner_anno = [n for n in names if n.endswith("Annotations.tar")][0]

        print("Extracting:", inner_img, "and", inner_anno)
        img_bytes = io.BytesIO(z.read(inner_img))
        ann_bytes = io.BytesIO(z.read(inner_anno))

        tmp_img = RAW_DIR / "101_ObjectCategories.tar.gz"
        tmp_ann = RAW_DIR / "Annotations.tar"
        with open(tmp_img, "wb") as f: f.write(img_bytes.getbuffer())
        with open(tmp_ann, "wb") as f: f.write(ann_bytes.getbuffer())

        # Extract tars into EXTRACT_DIR
        safe_extract_tar(tmp_img, EXTRACT_DIR)
        safe_extract_tar(tmp_ann, EXTRACT_DIR)

        # Clean temp
        tmp_img.unlink(missing_ok=True)
        tmp_ann.unlink(missing_ok=True)

print("Ready at:", EXTRACT_DIR,
      "| contains:", (EXTRACT_DIR / "101_ObjectCategories").exists(),
                     (EXTRACT_DIR / "Annotations").exists())


Extracting: caltech-101/101_ObjectCategories.tar.gz and caltech-101/Annotations.tar
Ready at: dataset/caltech101 | contains: True True


In [ ]:
dataset = torchvision.datasets.Caltech101(
    './dataset',
    download=True,
    transform = torchvision.transforms.Compose([
        torchvision.transforms.PILToTensor(),
        torchvision.transforms.ConvertImageDtype(torch.float),
        torchvision.transforms.Resize((224, 224)),
        torchvision.transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x)
    ])
)
n_samples = len(dataset)
n_train_samples = int(.8 * n_samples)
n_val_samples = int(.1 * n_samples)
n_test_samples = n_samples - n_train_samples - n_val_samples

train_ds, val_ds, test_ds = torch.utils.data.random_split(dataset, [
    n_train_samples, n_val_samples, n_test_samples
])

train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=32)
test_dl = torch.utils.data.DataLoader(test_ds, batch_size=32)

With the dataset ready, it is now time to adapt the model architecture in order to fit our needs. Define a new classifier for the AlexNet model having the same structure, changing only the number of output neurons to 101.

In [ ]:
last_in_features = model.classifier[-1].in_features
model.classifier[-1] = torch.nn.Linear(last_in_features, NUM_CLASSES)

### Training the model

Define an Adam optimizer with a learining rate of 1e-4 and a cross entropy loss. Afterwards, train the model for 2 epochs. Note the results

In [ ]:
print(model.features)
print(model.classifier)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
  (1): ReLU(inplace=True)
  (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (4): ReLU(inplace=True)
  (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU(inplace=True)
  (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): ReLU(inplace=True)
  (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
)
Sequential(
  (0): Dropout(p=0.5, inplace=False)
  (1): Linear(in_features=9216, out_features=4096, bias=True)
  (2): ReLU(inplace=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=4096, out_features=4096, bias=True

## Experiments:

1. Rerun training (restart kernel and run all cells) but this time, when loading the model in the first block of code, specify 'pretrained = True' in order to make use of the weights pretrained on Imagenet.
2. Rerun the code using the pretrained model but this time use a learning rate of 1e-3. What happens?
3. Rerun using the pretrained model and a lr of 1e-4 but this time only change the last layer in the model instead of the entire classifier.
3. Rerun the code using the pretrained model and a lr of 1e-4. This time, freeze the pretrained layers and only update the new layers for the first epochs. Afterwards, proceed to update the entire model. You can freeze parameters by specifying 'requires_grad = False'.
4. Rerun experiment 3 but gradually unfreeze layers instead of unfreezeing the entire model at once.

In [ ]:
def load_alexnet_model(pretrained=False):
    model = torchvision.models.alexnet(pretrained=pretrained)
    return model

In [ ]:
def prepare_caltech101_dataset(root_dir='./dataset'):
    transform = torchvision.transforms.Compose([
        torchvision.transforms.PILToTensor(),
        torchvision.transforms.ConvertImageDtype(torch.float),
        torchvision.transforms.Resize((224, 224)),
        torchvision.transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x)
    ])

    dataset = torchvision.datasets.Caltech101(
        root_dir,
        download=False,
        transform=transform
    )

    n_samples = len(dataset)
    n_train_samples = int(.8 * n_samples)
    n_val_samples = int(.1 * n_samples)
    n_test_samples = n_samples - n_train_samples - n_val_samples

    train_ds, val_ds, test_ds = torch.utils.data.random_split(dataset, [
        n_train_samples, n_val_samples, n_test_samples
    ])

    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
    val_dl = torch.utils.data.DataLoader(val_ds, batch_size=32)
    test_dl = torch.utils.data.DataLoader(test_ds, batch_size=32)

    return train_dl, val_dl, test_dl

In [ ]:
def iterate_dataloader(dataloader, model, criterion, optimizer=None, is_train=False, device=None):
    running_loss = 0.0
    correct = 0
    total = 0

    if is_train:
        model.train()
    else:
        model.eval()

    with torch.set_grad_enabled(is_train):
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return running_loss, correct, total

In [ ]:
def train_model(model, train_dl, val_dl, optimizer, criterion, num_epochs=2):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    for epoch in range(num_epochs):
        running_loss, _, _ = iterate_dataloader(
            train_dl, model, criterion, optimizer=optimizer, is_train=True, device=device)
        epoch_loss = running_loss / len(train_dl.dataset)
        print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {epoch_loss:.4f}")
        running_loss, correct, total = iterate_dataloader(
            val_dl, model, criterion, is_train=False, device=device)
        accuracy = 100 * correct / total
        print(f"Epoch {epoch + 1}/{num_epochs}, Validation Accuracy: {accuracy:.2f}%")

    print("Finished Training")
    return model

In [ ]:
def evaluate_model(model, test_dl, criterion):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    running_loss, correct, total = iterate_dataloader(
        test_dl, model, criterion, is_train=False, device=device)
    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

## Experiment 1: train with pretrained weights

In [ ]:
model = load_alexnet_model(pretrained=True)

last_in_features = model.classifier[-1].in_features
model.classifier[-1] = torch.nn.Linear(last_in_features, NUM_CLASSES)

train_dl, val_dl, test_dl = prepare_caltech101_dataset()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

trained_model = train_model(model, train_dl, val_dl, optimizer, criterion, num_epochs=2)

evaluate_model(trained_model, test_dl, criterion)

Epoch 1/2, Training Loss: 1.4739
Epoch 1/2, Validation Accuracy: 79.24%
Epoch 2/2, Training Loss: 0.3790
Epoch 2/2, Validation Accuracy: 84.54%
Finished Training
Test Accuracy: 85.50%


85.5005753739931

## Experiment 2: train with higher learning rate


In [ ]:
model = load_alexnet_model(pretrained=True)

last_in_features = model.classifier[-1].in_features
model.classifier[-1] = torch.nn.Linear(last_in_features, NUM_CLASSES)

train_dl, val_dl, test_dl = prepare_caltech101_dataset()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

trained_model = train_model(model, train_dl, val_dl, optimizer, criterion, num_epochs=2)

evaluate_model(trained_model, test_dl)

Epoch 1/2 [Training]: 100%|██████████| 217/217 [00:31<00:00,  6.79it/s, loss=3.32]


Epoch 1/2, Training Loss: 3.3237


Epoch 1/2 [Validation]: 100%|██████████| 28/28 [00:02<00:00, 10.03it/s]


Epoch 1/2, Validation Accuracy: 43.37%


Epoch 2/2 [Training]: 100%|██████████| 217/217 [00:31<00:00,  6.87it/s, loss=2.49]


Epoch 2/2, Training Loss: 2.4948


Epoch 2/2 [Validation]: 100%|██████████| 28/28 [00:02<00:00,  9.80it/s]


Epoch 2/2, Validation Accuracy: 50.75%
Finished Training
Test Accuracy: 48.68%


48.67663981588032

## Experiment 3: train by changing only the last layer


In [ ]:
model = load_alexnet_model(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier[-1].parameters():
    param.requires_grad = True

last_in_features = model.classifier[-1].in_features
model.classifier[-1] = torch.nn.Linear(last_in_features, NUM_CLASSES)

for param in model.classifier[-1].parameters():
    param.requires_grad = True


train_dl, val_dl, test_dl = prepare_caltech101_dataset()

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

criterion = torch.nn.CrossEntropyLoss()

trained_model = train_model(model, train_dl, val_dl, optimizer, criterion, num_epochs=2)

evaluate_model(trained_model, test_dl)

Epoch 1/2 [Training]: 100%|██████████| 217/217 [00:24<00:00,  8.77it/s, loss=2.38]


Epoch 1/2, Training Loss: 2.3784


Epoch 1/2 [Validation]: 100%|██████████| 28/28 [00:03<00:00,  8.07it/s]


Epoch 1/2, Validation Accuracy: 71.40%


Epoch 2/2 [Training]: 100%|██████████| 217/217 [00:24<00:00,  8.76it/s, loss=1.08]


Epoch 2/2, Training Loss: 1.0838


Epoch 2/2 [Validation]: 100%|██████████| 28/28 [00:02<00:00,  9.87it/s]


Epoch 2/2, Validation Accuracy: 77.16%
Finished Training
Test Accuracy: 76.64%


76.6398158803222

## Experiment 4: gradual unfreezing

In [ ]:
model = load_alexnet_model(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

last_in_features = model.classifier[-1].in_features
model.classifier[-1] = torch.nn.Linear(last_in_features, NUM_CLASSES)
for param in model.classifier[-1].parameters():
    param.requires_grad = True

train_dl, val_dl, test_dl = prepare_caltech101_dataset()

criterion = torch.nn.CrossEntropyLoss()

In [ ]:
unfreeze_stages = [
    {'name': 'classifier_last', 'layers': [model.classifier[-1]]},
    {'name': 'classifier_block2', 'layers': [model.classifier[3], model.classifier[4], model.classifier[5]]},
    {'name': 'classifier_block1', 'layers': [model.classifier[0], model.classifier[1], model.classifier[2]]},
    {'name': 'features_block5', 'layers': [model.features[10], model.features[11], model.features[12]]},
    {'name': 'features_block4', 'layers': [model.features[8], model.features[9]]},
    {'name': 'features_block3', 'layers': [model.features[6], model.features[7]]},
    {'name': 'features_block2', 'layers': [model.features[3], model.features[4], model.features[5]]},
    {'name': 'features_block1', 'layers': [model.features[0], model.features[1], model.features[2]]},
]

epochs_per_stage = 2
learning_rate = 1e-4

print("Starting gradual unfreezing training...")

for stage in unfreeze_stages:
    stage_name = stage['name']
    layers_to_unfreeze = stage['layers']

    print(f"\n--- Unfreezing stage: {stage_name} ---")

    for layer in layers_to_unfreeze:
        for param in layer.parameters():
            param.requires_grad = True

    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)

    print(f"Training for {epochs_per_stage} epochs with {stage_name} unfrozen...")
    trained_model = train_model(model, train_dl, val_dl, optimizer, criterion, num_epochs=epochs_per_stage)

print("\n--- Gradual unfreezing training finished ---")

print("\nEvaluating the final model on the test set...")
evaluate_model(trained_model, test_dl)

Starting gradual unfreezing training...

--- Unfreezing stage: classifier_last ---
Training for 2 epochs with classifier_last unfrozen...


Epoch 1/2 [Training]: 100%|██████████| 217/217 [00:24<00:00,  8.77it/s, loss=2.37]


Epoch 1/2, Training Loss: 2.3743


Epoch 1/2 [Validation]: 100%|██████████| 28/28 [00:02<00:00,  9.67it/s]


Epoch 1/2, Validation Accuracy: 69.09%


...

Epoch 2/2 [Training]: 100%|██████████| 217/217 [00:31<00:00,  6.82it/s, loss=0.0588]


Epoch 2/2, Training Loss: 0.0588


Epoch 2/2 [Validation]: 100%|██████████| 28/28 [00:02<00:00, 10.02it/s]


Epoch 2/2, Validation Accuracy: 85.35%
Finished Training

--- Gradual unfreezing training finished ---

Evaluating the final model on the test set...
Test Accuracy: 86.08%


86.07594936708861